# Dataset Processing

This notebook prepares the canonical training dataset from the raw CSV files in `datasets/`. It validates the source schema, merges the files, removes blank rows and exact duplicates, and saves both the raw merge and the cleaned combined CSV.


## Environment Setup

Install the notebook dependencies and import the libraries used for dataset preparation.


In [ ]:
# Only install packages that Colab does not ship with.
# Do NOT reinstall torch / torchvision — Colab's pre-installed versions are
# already built against the same CUDA toolkit.  Upgrading one without the
# other causes the CUDA-major-version mismatch error.
%pip install -q transformers datasets accelerate

In [ ]:
from pathlib import Path

import pandas as pd


## Project Configuration

Define the dataset paths, expected schema, and output files for the cleaned dataset artifact.


In [ ]:
# Set the default folder paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "datasets"

RAW_COMBINED_PATH = DATA_DIR / "combined_dataset_raw.csv"
COMBINED_INPUT_PATH = DATA_DIR / "combined_dataset.csv"

EXPECTED_COLUMNS = ["text", "label", "persona", "source_type", "scenario"]
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"
EXPECTED_SOURCE_FILE_COUNT = 6


## Dataset Assembly

Find the raw CSV sources, validate their columns, merge them, and save both the raw merged file and the cleaned training dataset.


In [ ]:
# Get the paths of all the datasets
csv_paths = sorted(DATA_DIR.glob("*.csv"))
source_paths = [
    path
    for path in csv_paths
    if path.name not in {RAW_COMBINED_PATH.name, COMBINED_INPUT_PATH.name}
]

# Raise error if the number of data files is not as expected
if len(source_paths) != EXPECTED_SOURCE_FILE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_SOURCE_FILE_COUNT} source CSV files, found {len(source_paths)}: "
        f"{[path.name for path in source_paths]}"
    )

In [ ]:
frames = []
for path in source_paths:
    frame = pd.read_csv(path)

    missing_columns = [col for col in EXPECTED_COLUMNS if col not in frame.columns]
    extra_columns = [col for col in frame.columns if col not in EXPECTED_COLUMNS]

    # Raise error if there are any columns missing
    if missing_columns:
        raise ValueError(f"{path.name} is missing required columns: {missing_columns}")

    # Warn for extra columns
    if extra_columns:
        print(f"Warning: {path.name} has extra columns that will be dropped: {extra_columns}")

    # Keep only specific columns
    frame = frame[EXPECTED_COLUMNS].copy()
    # Add another column
    frame["source_file"] = path.name
    # Add to array
    frames.append(frame)

# Save the Raw Data as CSV
combined_raw_df = pd.concat(frames, ignore_index=True)
combined_raw_df.to_csv(RAW_COMBINED_PATH, index=False)

In [ ]:
# Convert info to strings and remove leading and trailing spaces
combined_df = combined_raw_df.copy()
combined_df[TEXT_COLUMN] = combined_df[TEXT_COLUMN].astype(str).str.strip()
combined_df[LABEL_COLUMN] = combined_df[LABEL_COLUMN].astype(str).str.strip()

# Remove rows with empty text or label
combined_df = combined_df.loc[
    combined_df[TEXT_COLUMN].ne("") & combined_df[LABEL_COLUMN].ne("")
]

# Remove doplicate rows
combined_df = combined_df.drop_duplicates(subset=EXPECTED_COLUMNS).reset_index(drop=True)

# Save updated CSV
combined_df.to_csv(COMBINED_INPUT_PATH, index=False)

merge_summary = pd.DataFrame(
    {
        "file": [path.name for path in source_paths],
        "rows": [len(frame) for frame in frames],
    }
)

display(merge_summary)
print(f"Raw merged rows: {len(combined_raw_df):,}")
print(f"Clean merged rows: {len(combined_df):,}")
print(f"Saved raw merge to: {RAW_COMBINED_PATH}")
print(f"Saved clean merge to: {COMBINED_INPUT_PATH}")

display(combined_df.head())
display(combined_df[LABEL_COLUMN].value_counts().rename_axis("label").to_frame("count"))
